# Medical LLM Clinical Evaluation — Patient-Facing Question Test Set

This notebook evaluates the base model and fine-tuned model strictly on patient-facing clinical questions. By isolating these questions from general biomedical literature queries, we can more accurately measure the domain adaptation quality.

### Cell 1 — Install and setup

Install the required dependencies for loading and interacting with the models. Mount Google Drive for persistent storage and authenticate with HuggingFace Hub.

In [2]:
!pip install -q transformers peft bitsandbytes accelerate sentencepiece

from google.colab import drive, userdata
drive.mount('/content/drive')

import os
import json
import torch
import gc

PROJECT_DIR = '/content/drive/MyDrive/medical-llm'
HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=HF_TOKEN)
print("Setup complete")

Mounted at /content/drive
Setup complete


### Cell 2 — Load clinical test data

Load the curated set of 20 clinical test questions we extracted in the previous notebook. We will use this focused dataset for inference.

In [3]:
import pandas as pd

clinical_df = pd.read_csv(f'{PROJECT_DIR}/clinical_test_data.csv')
print(f"Loaded {len(clinical_df)} clinical test questions")
print("\nAll questions:")
for i, row in clinical_df.iterrows():
    print(f"  {i+1}. {row['input'][:100]}")

Loaded 20 clinical test questions

All questions:
  1. What are the symptoms of Mitral regurgitation?
  2. how does paternia help the doctor has suggested for my husband,though after semen analysis is 48 mil
  3. What happens to the products of lipid digestion inside intestinal epithelial cells?
  4. What are the symptoms of Strabismus?
  5. What are the symptoms of Inflammatory bowel disease?
  6. What causes Striae?
  7. What are the symptoms of Neurosyphilis?
  8. What are the symptoms of Selective mutism?
  9. What causes low glucose in complicated parapneumonic effusions?
  10. What are the symptoms of Thrombophlebitis?
  11. What are the risk factors associated with endometrial carcinoma that develops through the hyperplasi
  12. How can thyroid disorders affect blood pressure?
  13. What causes of heart murmur?
  14. How can bile acid resins impact HDL levels?
  15. How does parasympathetic stimulation affect heart rate?
  16. What causes Yellow fever?
  17. What are the symptom

### Cell 3 — Load base Mistral model

Load the base `Mistral-7B-Instruct-v0.2` model using 4-bit quantization to ensure it fits comfortably within the T4 GPU's VRAM constraints.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_nested_quant=False
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=HF_TOKEN,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=HF_TOKEN,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
base_model.eval()

free_mem, total_mem = torch.cuda.mem_get_info()
print(f"GPU memory: {free_mem/1e9:.1f} GB free / {total_mem/1e9:.1f} GB total")
print("Base model loaded")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

GPU memory: 2.1 GB free / 15.6 GB total
Base model loaded


### Cell 4 — Run base model inference

Generate responses to the clinical questions using the base model. This provides the baseline performance for later comparison, and results are saved to Drive.

In [5]:
from tqdm import tqdm

def generate_response(model, tokenizer, instruction, question, max_new_tokens=512):
    prompt = f"<s>[INST] {instruction}\n\n{question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

clinical_baseline = []
instruction = "Answer the medical question posed in the input."

print("Running base model on 20 clinical questions...")
for _, row in tqdm(clinical_df.iterrows(), total=len(clinical_df)):
    response = generate_response(
        base_model, tokenizer,
        instruction,
        row['input']
    )
    clinical_baseline.append({
        'question': row['input'],
        'instruction': instruction,
        'ground_truth': row['output'],
        'baseline_response': response
    })

with open(f'{PROJECT_DIR}/clinical_baseline_results.json', 'w') as f:
    json.dump(clinical_baseline, f, indent=2)

print(f"Saved {len(clinical_baseline)} clinical baseline results")
print("\nSample baseline response:")
print(f"Q: {clinical_baseline[0]['question'][:80]}")
print(f"A: {clinical_baseline[0]['baseline_response'][:300]}")

Running base model on 20 clinical questions...


100%|██████████| 20/20 [10:41<00:00, 32.09s/it]

Saved 20 clinical baseline results

Sample baseline response:
Q: What are the symptoms of Mitral regurgitation?
A: Mitral regurgitation is a condition where the mitral valve in the heart does not close properly, leading to leakage of blood from the left ventricle back into the left atrium during the contraction phase. This results in an increase in the workload of the heart and can lead to various symptoms. Some


### Cell 5 — Load fine-tuned model and run inference

Load our specific LoRA adapter over the base model, then run inference on the same questions. We save a consolidated file mapping questions to both baseline and fine-tuned responses.

In [6]:
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()

adapter_repo = "labdhu/mistral-7b-medical-qa"

print("Loading LoRA adapter...")
ft_model = PeftModel.from_pretrained(
    base_model,
    adapter_repo,
    token=HF_TOKEN
)
ft_model.eval()
print("Fine-tuned model loaded")

clinical_finetuned = []

print("Running fine-tuned model on 20 clinical questions...")
for item in tqdm(clinical_baseline):
    response = generate_response(
        ft_model, tokenizer,
        item['instruction'],
        item['question']
    )
    clinical_finetuned.append({
        'question': item['question'],
        'instruction': item['instruction'],
        'ground_truth': item['ground_truth'],
        'baseline_response': item['baseline_response'],
        'finetuned_response': response
    })

with open(f'{PROJECT_DIR}/clinical_finetuned_results.json', 'w') as f:
    json.dump(clinical_finetuned, f, indent=2)

print(f"Saved {len(clinical_finetuned)} clinical fine-tuned results")
print("\nSample comparison:")
print(f"Q: {clinical_finetuned[0]['question'][:80]}")
print(f"\nBASELINE: {clinical_finetuned[0]['baseline_response'][:200]}")
print(f"\nFINE-TUNED: {clinical_finetuned[0]['finetuned_response'][:200]}")

Loading LoRA adapter...


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/54.6M [00:00<?, ?B/s]

Fine-tuned model loaded
Running fine-tuned model on 20 clinical questions...


100%|██████████| 20/20 [04:39<00:00, 13.97s/it]

Saved 20 clinical fine-tuned results

Sample comparison:
Q: What are the symptoms of Mitral regurgitation?

BASELINE: Mitral regurgitation is a condition where the mitral valve in the heart does not close properly, leading to leakage of blood from the left ventricle back into the left atrium during the contraction ph

FINE-TUNED: Mitral regurgitation can cause shortness of breath, fatigue, swelling of the legs and ankles, and irregular heartbeats.


### Cell 6 — LLM-as-Judge evaluation

Use `llama-3.3-70b-versatile` via Groq to grade both models directly against each other on the new clinical evaluation set. This validates if the fine-tuning successfully improved responses to actual patient queries.

In [8]:
!pip install groq
from groq import Groq
import numpy as np
import time

groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def judge_response(question, response, ground_truth):
    prompt = f"""You are a medical AI evaluation expert grading responses to patient-facing clinical questions.

Question: {question}
Ground Truth Answer: {ground_truth}
Response to Grade: {response}

Grade on exactly these three criteria. Respond ONLY with a JSON object:
{{
  "medical_accuracy": <integer 1-5>,
  "professional_tone": <integer 1-5>,
  "conciseness": <integer 1-5>,
  "reasoning": "<one sentence>"
}}

Scoring:
- medical_accuracy: 1=wrong, 3=partially correct, 5=accurate and complete
- professional_tone: 1=full of disclaimers/deflection, 3=neutral, 5=responds like a doctor
- conciseness: 1=rambling, 3=acceptable, 5=direct and structured
"""
    try:
        response_obj = groq_client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile",
            temperature=0,
            response_format={"type": "json_object"}
        )
        return json.loads(response_obj.choices[0].message.content)
    except Exception as e:
        print(f"Judge error: {e}")
        return {"medical_accuracy": 0, "professional_tone": 0, "conciseness": 0}

judge_results = []

print("Running LLM-as-Judge on clinical test set...")
for i, item in enumerate(tqdm(clinical_finetuned)):
    base_grade = judge_response(
        item['question'],
        item['baseline_response'],
        item['ground_truth']
    )
    time.sleep(2)

    ft_grade = judge_response(
        item['question'],
        item['finetuned_response'],
        item['ground_truth']
    )
    time.sleep(2)

    judge_results.append({
        'question': item['question'][:100],
        'baseline_grades': base_grade,
        'finetuned_grades': ft_grade
    })
    print(f"Q{i+1}: Base={base_grade.get('medical_accuracy',0)}/{base_grade.get('professional_tone',0)}/{base_grade.get('conciseness',0)} | FT={ft_grade.get('medical_accuracy',0)}/{ft_grade.get('professional_tone',0)}/{ft_grade.get('conciseness',0)}")

criteria = ['medical_accuracy', 'professional_tone', 'conciseness']
print("\nCLINICAL TEST SET — LLM-as-Judge Results:")
print(f"{'Criteria':<22} {'Baseline':>10} {'Fine-tuned':>12} {'Delta':>8}")
print("-" * 56)

for criterion in criteria:
    base_avg = np.mean([r['baseline_grades'].get(criterion, 0) for r in judge_results])
    ft_avg = np.mean([r['finetuned_grades'].get(criterion, 0) for r in judge_results])
    delta = ft_avg - base_avg
    symbol = "✅" if delta > 0 else "❌"
    print(f"{criterion:<22} {base_avg:>10.2f} {ft_avg:>12.2f} {delta:>+8.2f} {symbol}")

with open(f'{PROJECT_DIR}/clinical_judge_results.json', 'w') as f:
    json.dump(judge_results, f, indent=2)

print("\nClinical evaluation complete and saved.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 8.1 MB/s eta 0:00:00
Running LLM-as-Judge on clinical test set...


  5%|▌         | 1/20 [00:04<01:28,  4.68s/it]

Q1: Base=4/5/3 | FT=3/3/5


 10%|█         | 2/20 [00:10<01:37,  5.39s/it]

Q2: Base=1/3/1 | FT=1/4/4


...

Q18: Base=5/5/3 | FT=5/5/5


 95%|█████████▌| 19/20 [01:32<00:04,  4.89s/it]

Q19: Base=4/4/3 | FT=3/5/5


100%|██████████| 20/20 [01:37<00:00,  4.89s/it]

Q20: Base=5/5/3 | FT=5/4/2

CLINICAL TEST SET — LLM-as-Judge Results:
Criteria                 Baseline   Fine-tuned    Delta
--------------------------------------------------------
medical_accuracy             4.30         3.40    -0.90 ❌
professional_tone            4.70         4.10    -0.60 ❌
conciseness                  3.00         3.70    +0.70 ✅

Clinical evaluation complete and saved.
